# LLM Factor Mining Agent

QuantaAlpha 式闭环：多方向假设 → 因子生成（硬约束） → IS 网格搜索 → 横向评审 → OOS 验证 → 研究轨迹进化。

主循环：`one_batch()` — 一个 batch 完成一次完整的研究迭代。

In [1]:
import requests
import json
import os
import re
import sys
import time

# Ensure local modules are discoverable
sys.path.insert(0, ".")

def load_env(path=".env"):
    """Load KEY=VALUE pairs from .env file (gitignored)."""
    if os.path.exists(path):
        with open(path, encoding="utf-8") as f:
            for line in f:
                line = line.strip()
                if not line or line.startswith("#") or "=" not in line:
                    continue
                k, v = line.split("=", 1)
                os.environ.setdefault(k.strip(), v.strip())

load_env()

API_KEY = os.environ.get("OPENCODE_GO_API_KEY", "")
if not API_KEY:
    raise RuntimeError("Missing OPENCODE_GO_API_KEY: set it in .env or environment")
BASE_URL = "https://opencode.ai/zen/go/v1"

HEADERS = {
    "Authorization": f"Bearer {API_KEY}",
    "Content-Type": "application/json",
}

MODEL = "deepseek-v4-pro"
print(f"Model: {MODEL}")

Model: deepseek-v4-pro


In [2]:
def chat(messages, model=None, max_tokens=8192, temperature=0.8):
    """Call OpenCode Go chat/completions API with retry on transient errors."""
    if model is None:
        model = MODEL

    data = {
        "model": model,
        "messages": messages,
        "max_tokens": max_tokens,
        "temperature": temperature,
    }

    retry_status = {429, 500, 502, 503, 504}
    max_retries = 5
    backoff = [2, 4, 8, 16, 32]

    for attempt in range(max_retries + 1):
        try:
            resp = requests.post(f"{BASE_URL}/chat/completions", headers=HEADERS, json=data, timeout=300)

            if resp.status_code == 200:
                result = resp.json()
                msg = result["choices"][0]["message"]
                content = msg.get("content", "") or ""
                reasoning = msg.get("reasoning_content", "") or ""
                usage = result.get("usage", {})

                return {
                    "content": content,
                    "reasoning": reasoning,
                    "answer": content if content.strip() else reasoning,
                    "usage": usage,
                    "model": result.get("model", ""),
                }

            if resp.status_code in retry_status and attempt < max_retries:
                wait = backoff[attempt]
                print(f"[retry] {resp.status_code} -> sleep {wait}s (attempt {attempt + 1}/{max_retries})")
                time.sleep(wait)
                continue

            raise RuntimeError(f"API error {resp.status_code}: {resp.text[:300]}")

        except (requests.exceptions.ConnectionError, requests.exceptions.Timeout) as e:
            if attempt < max_retries:
                wait = backoff[attempt]
                print(f"[retry] {type(e).__name__} -> sleep {wait}s (attempt {attempt + 1}/{max_retries})")
                time.sleep(wait)
                continue
            raise RuntimeError(f"API connection error: {e}")

    raise RuntimeError(f"API error: retries exhausted after {max_retries} attempts")


def quick_test():
    """Verify API connectivity."""
    resp = requests.get(f"{BASE_URL}/models", headers=HEADERS)
    if resp.status_code == 200:
        models = [m["id"] for m in resp.json().get("data", [])]
        print(f"[OK] {len(models)} models available: {models[:5]}...")
        result = chat([{"role": "user", "content": "Say hi."}], max_tokens=10)
        print(f"[OK] Tokens: in={result['usage'].get('prompt_tokens')}, out={result['usage'].get('completion_tokens')}")
    else:
        print(f"[FAIL] Models endpoint: {resp.status_code}")

quick_test()

[OK] 26 models available: ['minimax-m3', 'minimax-m2.7', 'minimax-m2.5', 'kimi-k3', 'kimi-k2.7-code']...
[OK] Tokens: in=86, out=10


## 初始化

回测引擎 + 研究轨迹（跨 batch 记忆）。

In [3]:
from backtest.engine import FactorBacktester
from agent.trajectory import ResearchTrajectory

# 先跑 BTCUSDT，后续扩展多币种
bt = FactorBacktester("data/SANDUSDT_1H.csv", commission_bps=6)
trajectory = ResearchTrajectory("trajectory.json")

print(f"Symbol: {bt.symbol}")
print(f"IS:     {bt.is_range} ({bt.is_rows} rows)")
print(f"OOS:    {bt.oos_range} ({bt.oos_rows} rows)")
print(f"Commission: {bt.commission_bps} bps")
print(f"轨迹方向数: {len(trajectory.data['directions'])}")

Symbol: SANDUSDT_1H
IS:     ('2021-01-25', '2025-06-29') (38778 rows)
OOS:    ('2025-06-29', '2026-06-29') (8760 rows)
Commission: 6 bps
轨迹方向数: 0


## Agent Batch 主循环

一个 batch：方向假设 → 因子生成 → IS网格搜索+代码筛选 → OOS → 轨迹更新。

筛选阈值在 `config.json`，改动后重新执行本 cell 生效。

In [4]:
from agent.hypothesis import generate_directions
from agent.factor_gen import generate_factor as gen_factor
from agent.judge import ask_oos_failure_analyst, ask_is_failure_analyst
from agent.config import load_config
from agent.screener import screen_factor, screen_oos

# 筛选阈值（改 config.json 即可调整）
cfg = load_config("config.json")

def one_batch(max_directions: int = 6) -> dict:
    """One full research iteration."""
    print("=" * 70)
    print("BATCH START")
    print(f"[筛选配置] {cfg}")

    # ---- Step 1: Hypothesis Agent ----
    print("\n[1] Hypothesis Agent: 生成研究方向...")
    hypo = generate_directions(chat, trajectory.summary(bt.symbol), symbol=bt.symbol)
    directions = hypo["directions"][:max_directions]
    if not directions:
        print("[FAIL] 方向解析失败")
        print(hypo["raw_response"][:500])
        return None
    for i, d in enumerate(directions):
        print(f"  [{i}] {d['name']} | {d['logic'][:60]}")

    # ---- Step 2: Factor Generation (with trajectory context) ----
    print("\n[2] Factor Generation...")
    factors = []
    for d in directions:
        trajectory.add_direction(d["name"], d["logic"], bt.symbol)
        ctx = trajectory.direction_context(d["name"], bt.symbol)
        f = gen_factor(chat, d, trajectory_context=ctx)
        status = "OK" if f["valid"] else f"FAIL ({f['violation_reason'][:40]})"
        print(f"  [{d['name']}] {status}, retries={f['retries']}")
        factors.append(f)
        if f["valid"]:
            neg = dict(f)
            neg["formula"] = f"-( {f['formula']} )"
            factors.append(neg)

    # ---- Step 3: IS Grid Search + Code Screening ----
    print("\n[3] IS Grid Search + Code Screening...")
    is_entries = {}
    for i, f in enumerate(factors):
        if not f["valid"]:
            continue
        try:
            result = bt.evaluate(f["formula"], f["category"])
        except Exception as e:
            print(f"  [{i}] {f['direction']['name']}: IS FAILED ({e})")
            continue
        isr = result["is_result"]

        # Sharpe 全负 -> LLM复盘 + 记录教训
        if isr["sharpe_max"] < 0:
            factor_info = f"{f['direction']['name']} | {f['formula']}"
            analysis = ask_is_failure_analyst(
                chat, result["is_report"], factor_info, "IS Sharpe全负")
            learning = "IS Sharpe全负。" + analysis["analysis"]
            if analysis["improve"]:
                learning += f" 改进: {analysis['improve']}"
            trajectory.add_attempt(
                f["direction"]["name"],
                bt.symbol,
                f["formula"],
                isr,
                {"params": [], "pass_rate": "0/0"},
                learning,
            )
            trajectory.update_status(f["direction"]["name"], "failed", bt.symbol)
            print(f"  [{i}] {f['direction']['name']}: Sharpe max={isr['sharpe_max']}<0")
            print(f"    复盘: {analysis['analysis'][:100]}")
            continue

        # 确定性筛选（替代LLM评审）
        screen = screen_factor(isr, cfg)
        print(f"  [{i}] {f['direction']['name']}: Sharpe max={isr['sharpe_max']}, "
              f"占比={isr['sharpe_positive_ratio']:.0%}, "
              f"入选{len(screen['selected_params'])}组, 通过={screen['passed']}")

        if not screen["passed"]:
            factor_info = f"{f['direction']['name']} | {f['formula']}"
            analysis = ask_is_failure_analyst(
                chat, result["is_report"], factor_info, screen["reason"])
            learning = f"{screen['reason']}。{analysis['analysis']}"
            if analysis["improve"]:
                learning += f" 改进: {analysis['improve']}"
            trajectory.add_attempt(
                f["direction"]["name"],
                bt.symbol,
                f["formula"],
                isr,
                {"params": screen["selected_params"], "pass_rate": "0/0"},
                learning,
            )
            trajectory.update_status(f["direction"]["name"], "failed", bt.symbol)
            print(f"    -> 记录教训: {screen['reason']}")
            print(f"    复盘: {analysis['analysis'][:100]}")
            if analysis["improve"]:
                print(f"    改进: {analysis['improve'][:100]}")
            continue

        stats = screen["selected_stats"]
        print(f"    -> 粗糙度={stats['roughness']['combined']}, "
              f"Sharpe范围={stats['sharpe_range']}")
        is_entries[i] = {
            "factor": f,
            "result": result,
            "selected_params": screen["selected_params"],
        }

    if not is_entries:
        print("[FAIL] 没有因子通过IS筛选")
        return None

    # ---- Step 4: OOS Test + trajectory update ----
    print(f"\n[4] OOS Test ({len(is_entries)} factors)...")
    oos_outcomes = []
    for i, entry in is_entries.items():
        f = entry["factor"]
        params = entry["selected_params"]
        is_result = entry["result"]
        try:
            oos = bt.evaluate_oos(f["formula"], params, is_result["is_result"])
        except Exception as e:
            print(f"  [{i}] {f['direction']['name']}: OOS FAILED ({e})")
            continue

        factor_info = f"{f['direction']['name']} | {f['formula']}"

        # ---- OOS 代码硬门槛 (2档) ----
        screen = screen_oos(oos["oos_result"], params, cfg)

        # OOS pass rate among selected params (记录用)
        oos_lookup = {(r["window"], r["threshold"]): r
                      for r in oos["oos_result"]["results"]}
        pass_count = sum(1 for w, th in params
                         if oos_lookup.get((w, th), {}).get("sharpe", 0) > 0)

        # OOS 达标参数 (Sharpe >= oos_sharpe_min)，记录并保存
        qualified = screen["qualified"]
        print(f"  [{i}] {f['direction']['name']}: 门槛判定={screen['passed']}, "
              f"OOS达标参数: {len(qualified)}/{len(params)}个 "
              f"(Sharpe >= {cfg['oos_sharpe_min']})")
        for w, th, s in qualified:
            print(f"      ({w}, {th}) Sharpe={s:.3f}")

        oos_summary = {"params": params,
                       "pass_rate": f"{pass_count}/{len(params)}",
                       "oos_qualified_params": [(w, th) for w, th, _ in qualified],
                       "oos_qualified_count": f"{len(qualified)}/{len(params)}"}

        if screen["passed"]:
            # 通过: 不调用LLM，固定文案
            learning = f"OOS通过: {len(qualified)}/{len(params)}参数OOS Sharpe>={cfg['oos_sharpe_min']}"
            status = "passed"
        else:
            # 失败: LLM复盘（对比IS/OOS报告找衰减原因）
            analysis = ask_oos_failure_analyst(
                chat,
                is_result["is_report"],
                oos["oos_report"],
                factor_info,
                screen["reason"],
            )
            learning = f"{screen['reason']}。{analysis['analysis']}"
            if analysis["improve"]:
                learning += f" 改进: {analysis['improve']}"
            status = "failed"
            print(f"    复盘: {analysis['analysis'][:120]}")
            if analysis["improve"]:
                print(f"    改进: {analysis['improve'][:120]}")

        trajectory.add_attempt(
            f["direction"]["name"],
            bt.symbol,
            f["formula"],
            is_result["is_result"],
            oos_summary,
            learning,
        )
        trajectory.update_status(
            f["direction"]["name"],
            status,
            bt.symbol,
        )

        oos_outcomes.append({
            "fidx": i,
            "params": params,
            "oos": oos,
            "screen": screen,
            "status": status,
        })

        print(f"  [{i}] {f['direction']['name']}: OOS {status}")

    print("\n" + "=" * 70)
    print("BATCH DONE")
    return {
        "directions": directions,
        "factors": factors,
        "is_entries": is_entries,
        "oos_outcomes": oos_outcomes,
    }


print("one_batch() ready.")

one_batch() ready.


## 运行一个 Batch

In [ ]:
batch = one_batch(max_directions=6)

BATCH START
[筛选配置] {'is_sharpe_min': 1.3, 'is_positive_ratio_min': 0.8, 'selected_roughness_max': 0.15, 'selected_trades_min': 50, 'selected_trades_max': 5000, 'min_selected_params': 3, 'oos_sharpe_min': 1.3, 'oos_positive_ratio_min': 0.8}

[1] Hypothesis Agent: 生成研究方向...
  [0] 趋势加速度与路径效率过滤 | 仅在短中期方向一致且趋势路径效率未进入末端状态时顺势，避免低质量趋势末段追价。
  [1] 波动率极值后的冷却反转 | 当已实现波动率进入滚动高分位且价格出现超买/超卖后，等待量能回落再反向，低波动状态不反转。
  [2] 量价同步爆发延续 | 当成交量扩张与收盘位置同向同步走强/走弱时顺势，不再用量价背离作为主信号。
  [3] 回撤修复质量与速度 | 用修复斜率、回补比例与修复期量能构建连续质量分，高质量修复看多，失败修复看空。
  [4] 过度反应与冷却确认 | 单根异常大K线后若放量但价格不再扩展，判为过度反应并反向；未冷却前不交易。
  [5] 主动买卖压力流的加速度 | 用收盘位置加权成交量的累积净主动买压的一阶变化与价格动能配合，同向加速时顺势、背离时过滤。

[2] Factor Generation...


c:\Users\HKCUSER\Desktop\Quant\strategy\crypto_cta\LLM_factor_mining\.\backtest\factor_lib.py:152: RuntimeWarning: invalid value encountered in divide
  np.asarray(x, dtype=np.float64) / np.asarray(y, dtype=np.float64),


  [趋势加速度与路径效率过滤] OK, retries=1
  [波动率极值后的冷却反转] OK, retries=0


c:\Users\HKCUSER\Desktop\Quant\strategy\crypto_cta\LLM_factor_mining\.\backtest\factor_lib.py:152: RuntimeWarning: invalid value encountered in divide
  np.asarray(x, dtype=np.float64) / np.asarray(y, dtype=np.float64),


  [量价同步爆发延续] OK, retries=0
  [回撤修复质量与速度] OK, retries=0
  [过度反应与冷却确认] OK, retries=2
  [主动买卖压力流的加速度] OK, retries=1

[3] IS Grid Search + Code Screening...
  [0] 趋势加速度与路径效率过滤: Sharpe max=-0.1674<0
    复盘: IS全网格Sharpe均为负、正收益占比0%，且参数粗糙度较低（0.153），说明不是参数过拟合或锯齿问题，而是因子方向或结构本身失效。公式中路径效率项使用 `(1 - ER)`，会在趋势顺畅时反而压
  [1] 趋势加速度与路径效率过滤: Sharpe max=0.3365, 占比=37%, 入选0组, 通过=False
    -> 记录教训: IS Sharpe最大值0.3365未达1.3
    复盘: 失败主因是因子内部方向与效率过滤逻辑反向：路径效率项使用了 `(1 - |RET_24|/Σ|RET_1|)`，趋势越流畅该项越接近0、信号越弱，趋势越震荡反而信号越强；叠加前置负号后，容易把“趋势一
    改进: 先将效率项改为 `eff = |RET_24| / Σ|RET_1|` 或对 `eff` 设置最低阈值，并对趋势高效时加权而非惩罚；同时去掉或反转前置负号，改为趋势方向自适应：20/60同向且向上做多
  [2] 波动率极值后的冷却反转: Sharpe max=-0.0916<0
    复盘: 该因子在IS全参数网格中Sharpe均为负、最大仅-0.09且Sharpe>0占比0%，说明不是参数过拟合或粗糙度问题，而是因子方向/构造逻辑整体失效；粗糙度0.1595表明它对参数不敏感地稳定亏损。
  [3] 波动率极值后的冷却反转: Sharpe max=0.853, 占比=76%, 入选0组, 通过=False
    -> 记录教训: IS Sharpe最大值0.853未达1.3
    复盘: IS峰值Sharpe仅0.853，出现在w=550/th=0.6的窄参数区；th<0.4时普遍为负，且阈值方向粗糙度0.3461偏高，说明信号对阈值敏感、低阈值噪声交易过多。公式中ts_rank只反映
    改进: 加入趋势过滤（如价格位于均线上方或动

## 批量运行 n 个 Batch

改 `N_BATCHES` / `MAX_DIRECTIONS` 后运行本 cell：连续跑 n 个 batch（带进度条），单个 batch 失败自动继续。

In [ ]:
# ==================== 参数（改这里） ====================
N_BATCHES = 5
MAX_DIRECTIONS = 5
# ========================================================

import time

try:
    from tqdm import tqdm
    iterator = tqdm(range(N_BATCHES), desc="Batches", unit="batch")
except ImportError:
    iterator = range(N_BATCHES)

ok = fail = 0
t0 = time.time()
for n in iterator:
    try:
        r = one_batch(max_directions=MAX_DIRECTIONS)
        if r is None:
            fail += 1
            print(f"[WARN] Batch {n + 1}/{N_BATCHES} 无产出，继续")
        else:
            ok += 1
    except KeyboardInterrupt:
        print("\n[STOP] 用户中断")
        break
    except Exception as e:
        fail += 1
        print(f"[ERROR] Batch {n + 1}/{N_BATCHES} 崩溃: {e}，继续")

print("=" * 70)
print(f"DONE: ok={ok}, failed={fail}, 耗时 {(time.time() - t0) / 60:.1f} min")
print("=" * 70)

Batches:   0%|          | 0/5 [00:00<?, ?batch/s]

BATCH START
[筛选配置] {'is_sharpe_min': 1.3, 'is_positive_ratio_min': 0.8, 'selected_roughness_max': 0.15, 'selected_trades_min': 50, 'selected_trades_max': 5000, 'min_selected_params': 3, 'oos_sharpe_min': 1.3, 'oos_positive_ratio_min': 0.8}

[1] Hypothesis Agent: 生成研究方向...
  [0] 趋势效率动量 | 用价格净位移与路径总波动之比识别流畅趋势，只在趋势效率高时顺势，避免震荡期假突破。
  [1] 极端偏离均值回复 | 价格在短窗口相对自适应均线或波动率带宽出现极端偏离后，捕捉其向中枢回归的反转机会。
  [2] 波动率聚集状态 | 利用已实现波动率或高低价区间的自相关特征，在高波动聚集期收紧入场条件、低波动扩张期侧重突破确认。
  [3] 量价动能背离 | 价格创区间新高或新低时，若成交量/能量指标未同步放大，视为趋势动能衰竭的背离信号。
  [4] 价格锚点偏离 | 度量价格相对近期高低点、整数关口或前收盘等心理锚点的偏离程度，捕捉锚定效应下的折返倾向。

[2] Factor Generation...


c:\Users\HKCUSER\Desktop\Quant\strategy\crypto_cta\LLM_factor_mining\.\backtest\factor_lib.py:152: RuntimeWarning: invalid value encountered in divide
  np.asarray(x, dtype=np.float64) / np.asarray(y, dtype=np.float64),


  [趋势效率动量] OK, retries=0


c:\Users\HKCUSER\Desktop\Quant\strategy\crypto_cta\LLM_factor_mining\.\backtest\factor_lib.py:152: RuntimeWarning: invalid value encountered in divide
  np.asarray(x, dtype=np.float64) / np.asarray(y, dtype=np.float64),


  [极端偏离均值回复] OK, retries=0
  [波动率聚集状态] OK, retries=0
  [量价动能背离] FAIL (参数占比50%，超过50%上限), retries=2
  [价格锚点偏离] OK, retries=0

[3] IS Grid Search + Code Screening...
  [0] 趋势效率动量: Sharpe max=0.4984, 占比=39%, 入选0组, 通过=False
    -> 记录教训: IS Sharpe最大值0.4984未达1.3
    复盘: 该因子IS Sharpe峰值仅0.4984，Sharpe>0占比39%、均值-0.0618，整体接近随机甚至偏负；正收益区仅窄幅集中在w≈300-450、th≈0.2附近，th从0.2到0.4迅速转负
    改进: 保留趋势效率逻辑，但增加过滤条件：要求中期趋势方向一致（如价格在更长均线同侧）且效率比率或ADX高于阈值，仅在趋势环境中开仓；同时对斜率用ATR或滚动波动率标准化，降低震荡市噪声。参数可重点测w=35
  [1] 趋势效率动量: Sharpe max=-0.2183<0
    复盘: 失败主因是因子方向反了：公式为 -slope×效率比率，使高效率上涨趋势得到负分、高效率下跌趋势得到正分，等价于逆趋势暴露；IS热力图Sharpe全部为负、最大值仅-0.218且正收益占比0%，粗糙度
  [2] 极端偏离均值回复: Sharpe max=-0.9642<0
    复盘: IS热力图全负且Sharpe>0占比为0%，粗糙度较低说明参数面平滑、不是参数过拟合，而是因子逻辑/方向存在系统性缺陷。该因子用120周期ts_rank对24周期偏离进行排序，信号明显滞后且只保留相对
  [3] 极端偏离均值回复: Sharpe max=-0.4068<0
    复盘: 该因子实际等价于 `ts_rank(zscore,120) - 0.5`，外层负号把“极端偏离均值回复”做反了，变成了追涨杀跌，因此IS Sharpe全参数为负、无正收益区域；粗糙度较低说明失败主要来
  [4] 波动率聚集状态: Sharpe max=-0.1202<0
    复盘: 失败主因不是参数没调好，而是因子结构本身：`(1 - ts_rank(VOL_ATR, 12

Batches:  20%|██        | 1/5 [21:43<1:26:52, 1303.00s/batch]

    -> 记录教训: IS Sharpe最大值0.331未达1.3
    复盘: IS网格中Sharpe均值-0.266、正收益占比仅18%，最大值0.331远低于1.3，且负值区域远大于正值区域，说明当前因子方向在样本内基本无效，甚至可能做反；正收益点零散、无成片稳健平台。因子本
    改进: 先验证反向：去掉最外层负号或改为做空高因子/做多低因子，确认是否只是符号用反；同时解耦20/120窗口，使用z-score或分位数标准化替代易贴边的min/max位置。加入趋势、波动率或成交量过滤，并
[FAIL] 没有因子通过IS筛选
[WARN] Batch 1/5 无产出，继续
BATCH START
[筛选配置] {'is_sharpe_min': 1.3, 'is_positive_ratio_min': 0.8, 'selected_roughness_max': 0.15, 'selected_trades_min': 50, 'selected_trades_max': 5000, 'min_selected_params': 3, 'oos_sharpe_min': 1.3, 'oos_positive_ratio_min': 0.8}

[1] Hypothesis Agent: 生成研究方向...
  [0] 量价动能背离 | 价格创区间新高或新低时，若成交量或累积能量指标未同步放大，视为趋势动能衰竭，据此反向或离场。
  [1] 处置效应回本抛压 | 用滚动成交量加权成本代理持仓成本，当价格从下方首次回升至成本密集区时，解套抛压易造成反弹受阻或二次探底。
  [2] 波动率压缩后突破确认 | 仅当已实现波动率从低分位持续收缩后首次扩张且价格突破近期高低点时顺势入场，避免低波动逆势放大信号。
  [3] K线买卖压力不平衡 | 将K线收盘在最高-最低区间的位置与成交量结合，衡量滚动窗口内主动买压与卖压的净失衡，沿占优压力方向交易。
  [4] 多尺度趋势一致性动量 | 当短、中、长多窗口标准化动量方向一致时才暴露趋势头寸，用多周期共振过滤单窗口噪声和震荡。

[2] Factor Generation...
  [量价动能背离] OK, retries=0
  [处置效应回本抛压] OK, retries=0
  [波动率压缩后突破确认] OK

## 筛选因子

从 trajectory.json 中过滤已通过的因子。

- `mode="IS pass"`：通过IS筛选（进入过OOS）的因子，含OOS通过的
- `mode="OOS pass"`：仅OOS通过的因子（至少1个参数 OOS Sharpe ≥ oos_sharpe_min）

In [7]:
import pandas as pd


def filter_factors(symbol: str = None, mode: str = "IS pass") -> pd.DataFrame:
    """
    从 trajectory 筛选因子。

    Args:
        symbol: 币种（如 "SANDUSDT_1H"），None = 全部币种
        mode: "IS pass"  = 通过IS筛选的因子（含OOS通过的）
              "OOS pass" = 仅OOS通过的因子
              （至少1个入选参数在OOS的Sharpe >= oos_sharpe_min）

    Returns:
        DataFrame 列: direction | symbol | hypothesis | formula |
                      is_sharpe_max | is_roughness | oos_pass_rate |
                      oos_qualified_count | learning
        按 is_sharpe_max 降序
    """
    rows = []
    for d in trajectory.data["directions"]:
        sym = d.get("symbol", "")
        if symbol and sym != symbol:
            continue
        for a in d["attempts"]:
            oos = a.get("oos_summary", {})
            pass_rate = oos.get("pass_rate", "0/0")

            is_pass = pass_rate != "0/0"  # 进过OOS = 通过IS筛选

            qual_str = oos.get("oos_qualified_count", "0/0")
            qual_n = int(qual_str.split("/")[0]) if "/" in qual_str else 0
            oos_pass = qual_n > 0

            if mode == "IS pass" and not is_pass:
                continue
            if mode == "OOS pass" and not (is_pass and oos_pass):
                continue

            rows.append({
                "direction": d["name"],
                "symbol": sym,
                "hypothesis": d.get("hypothesis", ""),
                "formula": a["formula"],
                "is_sharpe_max": a["is_summary"].get("sharpe_max"),
                "is_roughness": a["is_summary"].get("roughness"),
                "oos_pass_rate": pass_rate,
                "oos_qualified_count": qual_str,
                "learning": a.get("learning", ""),
            })

    df = pd.DataFrame(rows)
    if not df.empty:
        df = df.sort_values("is_sharpe_max", ascending=False).reset_index(drop=True)
    return df


# ---- 用法 ----
df_is = filter_factors(symbol="SANDUSDT_1H", mode="IS pass")
print(f"IS pass: {len(df_is)} 个因子")
display(df_is)

df_oos = filter_factors(symbol="SANDUSDT_1H", mode="OOS pass")
print(f"OOS pass: {len(df_oos)} 个因子")
display(df_oos)

IS pass: 0 个因子


""


OOS pass: 0 个因子


""
